## Validation and Fields - Follow-Along Practice

This notebook covers field constraints and custom validators in Pydantic.

#### Install Dependencies

Run the following cell to install Pydantic and Email-Validator (for email validation) in your notebook environment.

In [ ]:
!pip install pydantic email-validator

#### 1. The Field Function

Import `Field` from pydantic and use it to add constraints like length and ranges.

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class User(BaseModel):
    name: str = Field(min_length=1, max_length=100)
    age: int = Field(gt=0, le=120)
    email: str

# Valid instantiation
user = User(name="Alice", age=30, email="alice@example.com")
print("Valid User:", user)

#### 2. String Constraints

Control string length and format using `min_length`, `max_length`, and regex `pattern`.

In [ ]:
class UserProfile(BaseModel):
    username: str = Field(min_length=3, max_length=20)
    bio: str = Field(max_length=500)
    website: str = Field(pattern=r"^https?://.*")

# Valid
profile = UserProfile(
    username="alice_dev",
    bio="Python developer",
    website="https://example.com"
)
print("Profile:", profile)

# Invalid - username too short
try:
    profile_invalid = UserProfile(username="ab", bio="Hi", website="https://x.com")
except ValidationError as e:
    print("
Validation Error:")
    print(e)

#### 3. Numeric Constraints

Control number ranges using `gt` (greater than), `ge` (greater than or equal), `lt` (less than), and `le` (less than or equal).

In [ ]:
class Product(BaseModel):
    name: str
    price: float = Field(gt=0)           # Greater than 0
    quantity: int = Field(ge=0)          # Greater than or equal to 0
    discount: float = Field(ge=0, le=1)  # Between 0 and 1

product = Product(
    name="Widget",
    price=29.99,
    quantity=100,
    discount=0.15
)
print("Product:", product)

#### 4. Default Values with Field

Set default values for fields while applying constraints simultaneously.

In [ ]:
class APIConfig(BaseModel):
    api_key: str
    model: str = Field(default="gpt-4")
    max_tokens: int = Field(default=1000, ge=1, le=4096)
    temperature: float = Field(default=0.7, ge=0, le=2)

# Only api_key required
config = APIConfig(api_key="sk-abc123")

print("Model:", config.model)        # gpt-4
print("Max Tokens:", config.max_tokens)   # 1000
print("Temperature:", config.temperature)  # 0.7

#### 5. Field Descriptions

Add `description` metadata to aid documentation and JSON schemas.

In [ ]:
class Order(BaseModel):
    order_id: str = Field(description="Unique order identifier")
    total: float = Field(gt=0, description="Order total in USD")
    items: int = Field(ge=1, description="Number of items in order")

# Descriptions appear in generated JSON Schema!

#### 6. Custom Validators

Implement custom validation logic using the `@field_validator` decorator.

In [ ]:
from pydantic import field_validator

class UserCustom(BaseModel):
    username: str
    
    @field_validator("username")
    def validate_username(cls, v):
        if " " in v:
            raise ValueError("Username cannot contain spaces")
        return v.lower()  # Normalize to lowercase

user = UserCustom(username="AliceSmith")
print("Username:", user.username)  # alicesmith

#### 7. Real-World Example

A mock payment form with field constraints.

In [ ]:
class PaymentForm(BaseModel):
    card_number: str = Field(min_length=16, max_length=16)
    expiry_month: int = Field(ge=1, le=12)
    expiry_year: int = Field(ge=2024)
    cvv: str = Field(min_length=3, max_length=4)
    amount: float = Field(gt=0, description="Amount in USD")
    currency: str = Field(default="USD", min_length=3, max_length=3)

payment = PaymentForm(
    card_number="1234567890123456",
    expiry_month=12,
    expiry_year=2025,
    cvv="123",
    amount=99.99
)
print("Payment Form:", payment)

#### 8. Common Patterns

Pydantic offers helpful built-in types for common formats.

#### Email Validation

In [ ]:
from pydantic import EmailStr

class UserEmail(BaseModel):
    email: EmailStr

valid_user = UserEmail(email="info@example.com")
print("EmailStr:", valid_user)

#### URL Validation

In [ ]:
from pydantic import HttpUrl

class Link(BaseModel):
    url: HttpUrl

valid_link = Link(url="https://docs.pydantic.dev")
print("HttpUrl:", valid_link)

#### Constrained Lists

In [ ]:
class OrderList(BaseModel):
    items: list[str] = Field(min_length=1)  # At least one item

try:
    empty_order = OrderList(items=[])
except ValidationError as e:
    print("Validation Error on empty list:")
    print(e)

#### 9. JSON Schema Generation

Generate full JSON Schema standard specifications directly from your models.

In [ ]:
import pprint

class UserSchema(BaseModel):
    name: str = Field(min_length=1, description="User's full name")
    age: int = Field(ge=0, description="User's age in years")

pprint.pprint(UserSchema.model_json_schema())